## **Using Pre-trained Models**

## Hugging Face Transformers Library

* **Introduction to the Ecosystem**
  * Central hub for pre-trained models, datasets, and tools
  * Over 120,000 models and 30,000 datasets available
  * Standardized API across model architectures
  * Components:
    * Transformers: Model implementations
    * Datasets: Data handling and processing
    * Tokenizers: Text preprocessing
    * Accelerate: Distributed training made simple

* **Installation and Setup**
  * Basic installation: `pip install transformers`
  * Full installation with dependencies:
    ```
    pip install transformers[torch]
    pip install datasets tokenizers
    ```
  * GPU acceleration requires compatible PyTorch/TensorFlow

* **Core Components**
  * `AutoModel`: Automatically selects appropriate architecture
  * `AutoTokenizer`: Handles text preprocessing
  * `pipeline()`: High-level API for common tasks
  * `Trainer`: Simplified fine-tuning and training

---

## Basic Inference Patterns

**The Pipeline API: A Simple Entry Point**

The Pipeline API provides the easiest way to get started with pre-trained models. It handles all the preprocessing and postprocessing for you, allowing you to focus on using the model rather than setting it up.

* **Pipeline API for Quick Tasks**

In [17]:
from transformers import pipeline

# Text classification example
classifier = pipeline("sentiment-analysis")
result = classifier("I love using transformers!")
print(f"Classification result: {result}")

# Text generation example
generator = pipeline("text-generation", model="gpt2")
result = generator("Once upon a time", max_length=50)
print(f"Generated text: {result[0]['generated_text']}")
# Output: "Once upon a time there was a young man who had been in the service of..."

# Question answering example
qa = pipeline("question-answering")
result = qa(question="Where do I live?", 
            context="My name is Claude and I live in the cloud.")
print(f"Answer: {result['answer']} (confidence: {result['score']:.4f})")
# Output: "Answer: in the cloud (confidence: 0.9875)"

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cpu


Classification result: [{'label': 'POSITIVE', 'score': 0.9994327425956726}]


Device set to use cpu
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
No model was supplied, defaulted to distilbert/distilbert-base-cased-distilled-squad and revision 564e9b5 (https://huggingface.co/distilbert/distilbert-base-cased-distilled-squad).
Using a pipeline without specifying a model name and revision in production is not recommended.


Generated text: Once upon a time it is customary for a child to be taught to speak English. A second term for a child may be made in order to include another child or group of kids in some time frame.


To learn how to teach English properly


Device set to use cpu


Answer: the cloud (confidence: 0.3359)


**Working with Specific Models: A Deeper Approach**

While the pipeline API is convenient, you often need more control over the model's behavior. Working directly with model and tokenizer objects gives you this flexibility. This section shows how to load models directly and process their inputs and outputs.

* **Working with Specific Models**

In [18]:
from transformers import AutoTokenizer, AutoModelForCausalLM

# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2")

# Tokenize input - convert text to numbers the model understands
inputs = tokenizer("Hello, I'm a language model", return_tensors="pt")
print("Tokenized input:", inputs)
# Output: {'input_ids': tensor([[...]]), 'attention_mask': tensor([[...]])}

# Generate output - the model creates new token sequences
outputs = model.generate(**inputs, max_length=50)

# Decode output - convert numbers back to readable text
result = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("Generated text:", result)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Tokenized input: {'input_ids': tensor([[15496,    11,   314,  1101,   257,  3303,  2746]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1]])}
Generated text: Hello, I'm a language modeler. I'm a programmer. I'm a writer. I'm a writer. I'm a writer. I'm a writer. I'm a writer. I'm a writer. I'm a writer. I


## Model Types and Selection

**Understanding Different Model Architectures**

Not all transformer models are the same - they have different architectures optimized for different tasks. Understanding these differences helps you select the right model for your specific needs.

* **Encoder Models**
  * Best for: Understanding tasks (classification, NER, embeddings)
  * These models process the entire input at once, creating rich representations of text
  * Examples:
    * BERT/RoBERTa: General-purpose understanding
    * DeBERTa: Enhanced with disentangled attention
    * ALBERT: Parameter-efficient BERT variant
  * Usage pattern:

In [19]:
from transformers import AutoTokenizer, AutoModel

# Load encoder model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

# Process some text
text = "I want to understand the meaning of this sentence."
inputs = tokenizer(text, return_tensors="pt")

# Get the embeddings (representations)
outputs = model(**inputs)

# The last_hidden_state contains a representation for each token
embeddings = outputs.last_hidden_state

# Often we want a single representation for the whole sentence
# We can use the [CLS] token (first token) or average all tokens
sentence_embedding = embeddings[0, 0, :]  # CLS token
print(f"Embedding shape: {sentence_embedding.shape}")

# This embedding can now be used for classification, clustering, etc.



Embedding shape: torch.Size([768])


* **Decoder Models**
  * Best for: Text generation tasks
  * These models generate text one token at a time, looking only at previous tokens
  * Examples:
    * GPT-2/GPT-Neo: General text generation
    * Llama 2/Mistral: Open-weight powerful LLMs
    * Phi-2: Compact but capable model
  * Usage pattern:

In [20]:
from transformers import AutoTokenizer, AutoModelForCausalLM

# Load decoder model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2")

# Prepare input prompt
prompt = "The best way to learn programming is to"
inputs = tokenizer(prompt, return_tensors="pt")

# Generate continuation
outputs = model.generate(
    **inputs, 
    do_sample=True,  # Use sampling for more diverse outputs
    max_length=50    # Control output length
)

# Convert to text
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"Generated text: {generated_text}")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Generated text: The best way to learn programming is to do it yourself."

To learn Python, you need four courses: C++, Python, or Javascript. Learning Python and Javascript together can be challenging, but it's worth it. For example, in


* **Encoder-Decoder Models**
  * Best for: Sequence-to-sequence tasks (translation, summarization)
  * These models combine both architectures - encoder processes input, decoder generates output
  * Examples:
    * T5/FLAN-T5: Text-to-text framework
    * BART: Denoising autoencoder
    * Pegasus: Specialized for summarization
  * Usage pattern:

In [21]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("t5-base")

# Prepare input with task prefix
text = "summarize: My name is Sarah and I live in London. I work as a software engineer and have been in the field for over 10 years. I specialize in web development and machine learning applications."
inputs = tokenizer(text, return_tensors="pt")

# Generate summary
outputs = model.generate(**inputs, max_length=50)

# Decode the summary
summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"Summary: {summary}")

Summary: i'm a software engineer and have been in the field for over 10 years . i specialize in web development and machine learning applications .


## Advanced Inference Techniques

**Fine-tuning Generation Parameters**

When generating text, controlling exactly how the model produces outputs is crucial. This section explains the various parameters that let you balance creativity, relevance, and quality.

* **Controlling Generation Parameters**

In [22]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2")

prompt = "Write a short poem about artificial intelligence:"
inputs = tokenizer(prompt, return_tensors="pt")

# Adjusting generation parameters for different effects
outputs = model.generate(
    **inputs,
    max_length=100,            # Maximum length of output sequence
    min_length=30,             # Minimum length of output sequence
    do_sample=True,            # Enable sampling (vs greedy decoding)
    temperature=0.7,           # Controls randomness (lower = more deterministic)
    top_k=50,                  # Limits vocabulary to top k tokens
    top_p=0.95,                # Nucleus sampling (probability mass to consider)
    num_return_sequences=3,    # Number of different sequences to return
    no_repeat_ngram_size=2     # Prevent repetition of n-grams
)

# Print all generated sequences
for i, output in enumerate(outputs):
    generated_text = tokenizer.decode(output, skip_special_tokens=True)
    print(f"Poem {i+1}:\n{generated_text}\n")

  # Output will include 3 different poems with the settings above

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Poem 1:
Write a short poem about artificial intelligence:

I was working with a coworker, he said, and the first thing that came to mind was a story about a kid who had come out of a high school in New Jersey. He was an engineering major, which is really good for a scientist. So we were talking about how the world is being made, how that's going to be possible, what's happening to the human race, so we wanted to know what it would take to make

Poem 2:
Write a short poem about artificial intelligence:

"The human brain is a machine with a complex system of neurons. It can read and interpret text, understand syntax, think and think. That's how it learns. And that's what makes it smarter than most computers."
.
,
/
[1] Wikipedia: Wikipedia


Poem 3:
Write a short poem about artificial intelligence:

A good day's sleep can make your life a lot easier.
 (Or use this as a motivation to write a long poem:



**Processing Multiple Inputs Efficiently**

When working with multiple inputs, processing them as a batch is much more efficient than one at a time. This section demonstrates how to handle batched inputs and outputs.

* **Batched Inference for Efficiency**

In [23]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2")

# Process multiple inputs at once
batch_texts = [
    "The future of AI is",
    "Machine learning helps us",
    "Neural networks can"
]

# Tokenize all inputs together
batch_inputs = tokenizer(batch_texts, padding=True, return_tensors="pt")
print("Batch input shape:", batch_inputs.input_ids.shape)
# Output: "Batch input shape: torch.Size([3, x])" where x is padded length

# Generate completions for all prompts at once
batch_outputs = model.generate(**batch_inputs, max_length=30)

# Decode all outputs
results = [tokenizer.decode(output, skip_special_tokens=True) 
            for output in batch_outputs]

# Print each completion
for prompt, completion in zip(batch_texts, results):
    print(f"Prompt: {prompt}")
    print(f"Completion: {completion}\n")

ValueError: Asking to pad but the tokenizer does not have a padding token. Please select a token to use as `pad_token` `(tokenizer.pad_token = tokenizer.eos_token e.g.)` or add a new pad token via `tokenizer.add_special_tokens({'pad_token': '[PAD]'})`.

**Making Models Fit on Limited Hardware**

Large language models require significant memory. Quantization reduces memory requirements by using lower precision for weights, making models accessible on consumer hardware.

* **Model Quantization for Speed/Memory**

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Configure 8-bit quantization
quantization_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_enable_fp32_cpu_offload=True
)

# Load model with quantization
model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-2-7b-chat-hf",
    device_map="auto",  # Automatically decide device placement
    quantization_config=quantization_config
)

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-2-7b-chat-hf")

# Model now uses much less GPU memory
print(f"Model loaded with quantization. Type: {model.__class__.__name__}")

# Use as normal (but with some minor quality trade-off)
prompt = "Explain how quantization works in machine learning:"
inputs = tokenizer(prompt, return_tensors="pt")
outputs = model.generate(**inputs, max_length=200)
result = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(result)

OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Llama-2-7b-chat-hf.
401 Client Error. (Request ID: Root=1-67eeb0ff-02432e5a09908c6c66c714f9;95121de2-150e-499b-a1f9-04ecfc4c0390)

Cannot access gated repo for url https://huggingface.co/meta-llama/Llama-2-7b-chat-hf/resolve/main/config.json.
Access to model meta-llama/Llama-2-7b-chat-hf is restricted. You must have access to it and be authenticated to access it. Please log in.

## Prompt Engineering Techniques

**Understanding Prompting Patterns**

How you phrase your requests to models dramatically affects their outputs. This section covers different prompting techniques and why they work.

* **Basic Prompting Patterns**
  * Zero-shot: Direct task description without examples

In [ ]:
from transformers import pipeline

# Zero-shot example - just asking directly
classifier = pipeline("zero-shot-classification")
result = classifier(
    "The food was absolutely delicious and the service was excellent.",
    candidate_labels=["positive", "negative", "neutral"]
)

print("Zero-shot classification:")
print(f"Text: {result['sequence']}")
print(f"Label: {result['labels'][0]} (confidence: {result['scores'][0]:.4f})")
# Output shows "positive" with high confidence

No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1 (https://huggingface.co/facebook/bart-large-mnli).
Using a pipeline without specifying a model name and revision in production is not recommended.
c:\Users\me\myenv\Lib\site-packages\huggingface_hub\file_download.py:144: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\me\.cache\huggingface\hub\models--facebook--bart-large-mnli. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.micro

Zero-shot classification:
Text: The food was absolutely delicious and the service was excellent.
Label: positive (confidence: 0.9864)


  * Few-shot: Providing examples before the task

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2")

# Few-shot example with demonstrations
few_shot_prompt = """
Text: "The food was terrible." Sentiment: Negative
Text: "I had a great time." Sentiment: Positive
Text: "The service was slow." Sentiment: Negative
Text: "The view was breathtaking." Sentiment: Positive
Text: "The room was overpriced." Sentiment: 
"""

inputs = tokenizer(few_shot_prompt, return_tensors="pt")
outputs = model.generate(**inputs, max_length=len(few_shot_prompt) + 20)
result = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Few-shot learning result:")
print(result)
# Model should complete with "Negative" based on patterns

* Chain-of-Thought: Encouraging step-by-step reasoning

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
    
tokenizer = AutoTokenizer.from_pretrained("gpt2-xl")  # Larger model for better reasoning
model = AutoModelForCausalLM.from_pretrained("gpt2-xl")

# Chain-of-thought prompt with reasoning example
cot_prompt = """
Question: If John has 5 apples and gives 2 to Mary, how many does he have left?
Answer: John starts with 5 apples. He gives 2 to Mary. So he has 5 - 2 = 3 apples left.

Question: If a train travels 120 miles in 2 hours, what is its speed?
Answer:
"""

inputs = tokenizer(cot_prompt, return_tensors="pt")
outputs = model.generate(**inputs, max_length=300, temperature=0.7)
result = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Chain-of-thought reasoning:")
print(result)
# Should show step-by-step reasoning for calculating speed

**Working with Chat and Instruction-Tuned Models**

Modern LLMs are often specifically trained to follow instructions and engage in conversations. This section explains how to properly format inputs for these models.

* **Instruction Tuning and Chat Templates**

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

# Load an instruction-tuned model
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-2-7b-chat-hf")
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-2-7b-chat-hf", 
                                            device_map="auto")

# Create a conversation using the proper format
messages = [
    {"role": "system", "content": "You are a helpful assistant who explains complex concepts simply."},
    {"role": "user", "content": "Explain quantum computing in simple terms"}
]

# Apply the model's chat template - this formats messages properly for this specific model
input_text = tokenizer.apply_chat_template(messages, return_tensors="pt")

# Generate response
outputs = model.generate(input_text, max_length=500)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Chat model response:")
print(response)
# Output shows an explanation of quantum computing

## Working with Model Outputs

**Creating and Using Text Embeddings**

Embeddings are numerical representations of text that capture semantic meaning. This section shows how to generate and use these representations for tasks like similarity comparison.

* **Extracting and Processing Embeddings**

In [ ]:
  from transformers import AutoTokenizer, AutoModel
  import torch
  import torch.nn.functional as F
  
  # Load a model designed for creating good embeddings
  tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
  model = AutoModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
  
  # Sample texts to compare
  texts = [
      "This is an example sentence that we want to encode.",
      "Each sentence is converted to a vector for comparison.",
      "Completely unrelated text about something else entirely."
  ]
  
  # Tokenize texts
  inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")
  
  # Get model outputs
  with torch.no_grad():
      outputs = model(**inputs)
  
  # Mean pooling - take average of all token embeddings
  attention_mask = inputs['attention_mask']
  input_mask_expanded = attention_mask.unsqueeze(-1).expand(outputs.last_hidden_state.size()).float()
  embeddings = torch.sum(outputs.last_hidden_state * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)
  
  # Normalize embeddings
  embeddings = F.normalize(embeddings, p=2, dim=1)
  
  # Now we can use these embeddings for similarity calculations
  print(f"Embedding shape: {embeddings.shape}")  # Should be [3, 384]
  
  # Calculate similarity between first and second sentence
  similarity_1_2 = F.cosine_similarity(embeddings[0].unsqueeze(0), embeddings[1].unsqueeze(0))
  
  # Calculate similarity between first and third sentence
  similarity_1_3 = F.cosine_similarity(embeddings[0].unsqueeze(0), embeddings[3].unsqueeze(0))
  
  print(f"Similarity between sentences 1 and 2: {similarity_1_2.item():.4f}")
  print(f"Similarity between sentences 1 and 3: {similarity_1_3.item():.4f}")
  # First pair should have higher similarity than second pair

**Generating Text with Real-Time Output**

When generating long texts, users often prefer to see output as it's generated rather than waiting for the entire process to complete. This section demonstrates how to stream generation results.

* **Handling Generation with Streaming**

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, TextIteratorStreamer
  from threading import Thread
  import time
  
  tokenizer = AutoTokenizer.from_pretrained("gpt2")
  model = AutoModelForCausalLM.from_pretrained("gpt2")
  
  prompt = "Write a story about a robot learning to feel emotions:"
  inputs = tokenizer(prompt, return_tensors="pt")
  
  # Create a streamer object that will yield output tokens one by one
  streamer = TextIteratorStreamer(tokenizer, skip_special_tokens=True)
  
  # Set up generation parameters
  generation_kwargs = dict(
      inputs, 
      streamer=streamer, 
      max_length=200,
      do_sample=True,
      temperature=0.7
  )
  
  # Run generation in a separate thread
  thread = Thread(target=model.generate, kwargs=generation_kwargs)
  thread.start()
  
  # Simulate printing tokens as they're generated
  print("Generating story (streamed output):")
  print(prompt, end="", flush=True)
  
  # Print each piece as it's generated (in real application)
  collected_text = ""
  for text in streamer:
      print(text, end="", flush=True)
      collected_text += text
      time.sleep(0.05)  # Small delay to simulate real-time display
  
  print("\n\nFull generated text:")
  print(prompt + collected_text)

## Practical Considerations and Best Practices

**Choosing the Right Model for Your Task**

With thousands of models available, selecting the appropriate one is critical. This section provides guidance on making this decision based on various factors.

* **Model Selection Criteria**
  * Task appropriateness:
    * Classification/Understanding → Encoder models (BERT family)
    * Text generation → Decoder models (GPT family)
    * Translation/Summarization → Encoder-decoder models (T5 family)
  
  * Size constraints:
    * Limited CPU only → DistilBERT, TinyBERT, MobileBERT
    * Consumer GPU → Models under 7B parameters (Phi-2, Mistral-7B)
    * High-end GPU → Larger models (Llama-2-13B, Falcon-40B)
  
  * License considerations:
    * Commercial use → Check model card for license details
    * Open weights → Llama 2, Falcon, Mistral, MPT
    * Research only → Some models restrict commercial applications
  
  * Domain specificity:
    * General text → Foundation models (GPT-2, BERT, T5)
    * Code → CodeLlama, StarCoder, CodeBERT
    * Biomedical → BioBERT, PubMedBERT, BioGPT
    * Finance → FinBERT, BloombergGPT

**Optimizing Memory Usage**

Large language models require significant memory. This section covers techniques to reduce memory usage when working with these models.

* **Memory Optimization Techniques**

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Example showing various memory optimizations

# 1. CPU Offloading - keep only active parts in GPU
model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-2-7b-chat-hf",
    device_map="auto",       # Automatically decide optimal device placement
    offload_folder="offload" # Where to store offloaded weights
)

# 2. Use efficient attention implementation
model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-2-7b-chat-hf",
    device_map="auto",
    attn_implementation="flash_attention_2"  # Much faster and memory-efficient
)

# 3. Low precision for inference
model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-2-7b-chat-hf",
    torch_dtype=torch.float16,  # Use half precision
    device_map="auto"
)

# 4. Model sharding with Accelerate
from accelerate import dispatch_model

model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-2-7b-chat-hf")
model = dispatch_model(model, device_map="balanced")  # Spread across available devices

print("Model loaded with memory optimization")

**Handling Errors and Ensuring Robustness**

In production environments, model inference needs to be reliable. This section shows how to handle errors gracefully and implement fallback strategies.

* **Error Handling and Robustness**

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
  import torch
  import time
  
  tokenizer = AutoTokenizer.from_pretrained("gpt2")
  model = AutoModelForCausalLM.from_pretrained("gpt2")
  
  def generate_with_fallback(prompt, max_attempts=3):
      """Generate text with fallback strategies if errors occur"""
      
      inputs = tokenizer(prompt, return_tensors="pt")
      attempt = 1
      
      # Start with optimal but more demanding generation settings
      generation_config = {
          "max_length": 150,
          "do_sample": True,
          "temperature": 0.7,
          "top_p": 0.95
      }
      
      while attempt <= max_attempts:
          try:
              print(f"Attempt {attempt} with config: {generation_config}")
              
              # Set a timeout for generation
              start_time = time.time()
              outputs = model.generate(**inputs, **generation_config)
              generation_time = time.time() - start_time
              
              result = tokenizer.decode(outputs[0], skip_special_tokens=True)
              print(f"Generation successful in {generation_time:.2f} seconds")
              return result
              
          except (RuntimeError, torch.cuda.OutOfMemoryError) as e:
              print(f"Error on attempt {attempt}: {e}")
              
              # Simplify generation parameters with each attempt
              if attempt == 1:
                  # Try with smaller output and no sampling
                  generation_config = {
                      "max_length": 100,
                      "do_sample": False
                  }
              elif attempt == 2:
                  # Last resort: minimal generation
                  generation_config = {
                      "max_length": 50,
                      "do_sample": False
                  }
              
              attempt += 1
      
      # If all attempts fail, return a simple message
      return f"Unable to generate text for: {prompt}"
  
  # Test the robust generation function
  prompts = [
      "Write a short story about:",  # Ambiguous prompt
      "Explain the theory of relativity simply:",  # Clear prompt
      "Generate a 10,000 word essay on:"  # Potentially problematic prompt
  ]
  
  for prompt in prompts:
      print(f"\nPrompt: {prompt}")
      result = generate_with_fallback(prompt)
      print(f"Result: {result[:100]}...")  # Show beginning of result

**Improving Performance with Caching**

Repeatedly loading models is inefficient. This section demonstrates how to implement caching strategies to speed up applications that use language models.

* **Caching for Performance**

In [24]:
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer, AutoModelForCausalLM
import os
import time

# Create cache directory
os.makedirs("./hf_cache", exist_ok=True)

# 1. Pre-download models to local storage
print("Downloading model to local cache...")
start_time = time.time()
model_path = snapshot_download(
    repo_id="gpt2",
    cache_dir="./hf_cache"
)
download_time = time.time() - start_time
print(f"Model downloaded in {download_time:.2f} seconds to {model_path}")

# 2. Load model from local cache (much faster)
print("\nLoading model from cache:")
start_time = time.time()
model = AutoModelForCausalLM.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(
    model_path,
    use_fast=True  # Use rust-based tokenizer for speed
)
load_time = time.time() - start_time
print(f"Model loaded in {load_time:.2f} seconds")

# 3. Implement application-level caching for repeated inputs
class TokenizerCache:
    def __init__(self, tokenizer, cache_size=100):
        self.tokenizer = tokenizer
        self.cache = {}
        self.cache_size = cache_size
    
    def encode(self, text, **kwargs):
        # Create a cache key from text and kwargs
        cache_key = str(text) + str(sorted(kwargs.items()))
        
        if cache_key in self.cache:
            print("Cache hit!")
            return self.cache[cache_key]
        
        # Compute tokenization
        result = self.tokenizer(text, **kwargs)
        
        # Add to cache, managing size
        if len(self.cache) >= self.cache_size:
            # Remove a random entry (could use LRU for better performance)
            self.cache.pop(next(iter(self.cache)))
        
        self.cache[cache_key] = result
        return result

# Create cached tokenizer
cached_tokenizer = TokenizerCache(tokenizer)

# Test with repeated queries
print("\nTesting tokenizer cache:")
for _ in range(3):
    start_time = time.time()
    tokens = cached_tokenizer.encode("This is a test of the tokenizer cache system", 
                                    return_tensors="pt")
    tokenize_time = time.time() - start_time
    print(f"Tokenization took {tokenize_time:.6f} seconds")

# The first call should be slower than subsequent cached calls

Fetching 26 files:   0%|          | 0/26 [00:00<?, ?it/s]Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
c:\Users\me\mye

Model downloaded in 237.18 seconds to ./hf_cache\models--gpt2\snapshots\607a30d783dfa663caf39e06633721c8d4cfcd7e

Loading model from cache:
Model loaded in 0.27 seconds

Testing tokenizer cache:
Tokenization took 0.000482 seconds
Cache hit!
Tokenization took 0.000007 seconds
Cache hit!
Tokenization took 0.000003 seconds
